# WiDS 2026 - Baseline Experiment\n\nInteractive notebook for training pipeline exploration.

In [ ]:
import sys
sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.features import remove_redundant, add_engineered, get_feature_set
from src.evaluation import horizon_brier_score, mean_brier_score, c_index
from src.models import CoxPH, RSF, GBSA, MultiHorizonXGB

## 1. Load Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train = add_engineered(remove_redundant(train))
test = add_engineered(remove_redundant(test))

feature_cols = get_feature_set(train, level="medium")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Train: {train.shape}, Test: {test.shape}")

## 2. Single Model Comparison

In [ ]:
X = train[feature_cols]
y_time = train[TIME_COL].values
y_event = train[EVENT_COL].values

models = {
    "CoxPH": CoxPH(penalizer=0.1, features=FEATURES_MEDIUM),
    "RSF": RSF(n_estimators=200, max_depth=5, min_samples_leaf=10),
    "GBSA": GBSA(n_estimators=100, max_depth=3, learning_rate=0.05),
    "XGB": MultiHorizonXGB(random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    model.fit(X, y_time, y_event)
    preds = model.predict_proba(X)
    brier = mean_brier_score(y_time, y_event, preds)
    ci = c_index(y_time, y_event, preds[12])
    results[name] = {"Brier": brier, "C-index": ci}
    print(f"{name:6s}  Brier={brier:.4f}  C-index={ci:.4f}")

pd.DataFrame(results).T

## 3. Run Full Pipeline

In [ ]:
from src.train import main
main()